# Introduction

This is a self-guided step-by-step Jupyter notebook guide that will show you how to run Python scripts saved on the
```src/precog-esgf-intake/scripts``` directory of the [precog-esgf-intake](https://github.com/precog-ocean/precog-esgf-intake) repository.
The scripts will be run from within this Jupyter notebook for illustrative purposes.

To run each ```.py``` script, simply run each cell in this notebook and follow the in-cell prompts as if you were working on any ```Terminal prompt```.

You can also run equivalent commands on any ```Terminal prompt``` with slightly different semantics (shown below).


## Step 0. Make sure you have ```Python >3.12``` and ```Jupyter Notebook``` installed

To start ```Jupyter Notebook``` follow the [installation instructions here](https://jupyter.org/install)


## Step 1. Get the program files and install dependencies

Follow the instructions on the [precog-esgf-intake landing page](https://github.com/precog-ocean/precog-esgf-intake).

This will show you how to clone the repository onto your machine and create a virtual environment called `.venv` in
which to install our software dependencies.


## Step 1.1 Register the project's python kernel so Jupyter Notebook recognises it

Navigate to the directory where you have precog-esfg-intake source files saved and run the following in your actual
terminal, not as ordinary Python code in a notebook cell:


```bash
cd ~/Desktop/precog-esgf-intake

.venv/bin/python3.12 -m pip install ipykernel
.venv/bin/python3.12 -m ipykernel install \
    --user \
    --name precog-esgf-intake \
    --display-name "Python (precog-esgf-intake)"
```

These commands register the existing environment as a Python 3.12 kernel that is then recognised within Jupyter
Notebook. Using `.venv/bin/python3.12` explicitly avoids ambiguity about which Python interpreter is being configured.

## Step 2. Run Jupyter Notebook and specify the right Python kernel

To open Jupyter Notebook, type the following in a new terminal window:

```bash
jupyter notebook
```
In Jupyter's kernel selector or **Kernel → Change Kernel** menu, select **Python (precog-esgf-intake)**. In VS Code,
use **Select Kernel** at the top of the notebook. Refresh the interface if the newly registered kernel is not listed.

Registering a kernel does not automatically switch this open notebook to it (so make sure you refresh and select).

## Step 3. Verify the environment and set the working directory

Run the next cell before any script. It checks that the kernel uses the project's environment and that the three
workflow scripts and `intake_esgf` is available in its packaged layout.

In [7]:
from pathlib import Path
import sys
import importlib.util

PROJECT_ROOT = Path.home() / "Desktop" / "precog-esgf-intake"
VENV_ROOT = PROJECT_ROOT / ".venv"

assert PROJECT_ROOT.is_dir(), f"Project directory not found: {PROJECT_ROOT}"
assert Path(sys.prefix).resolve() == VENV_ROOT.resolve(), (
    f"Wrong kernel environment: {sys.prefix}\n"
    f"Select Python (precog-esgf-intake), which should use {VENV_ROOT}."
)

%cd {PROJECT_ROOT}

for name in (
    "intake_CatalogueSearch.py",
    "intake_OceanVarsDL.py",
    "intake_CellMeasuresDL.py",
):
    script = PROJECT_ROOT /"src"/"precog_esgf_intake"/"scripts" / name
    assert script.is_file(), f"Script not found: {script}"

assert importlib.util.find_spec("intake_esgf") is not None, (
    "intake_esgf is missing from the selected environment. "
    "Install the repository's declared dependencies into .venv, then restart the kernel."
)

print("Python:", sys.executable)
print("Environment:", sys.prefix)
print("Working directory:", Path.cwd())
%pip install -q -e .
import precog_esgf_intake
print(f'Imported local packaged layout for precog-esgf-intake, which points to: {precog_esgf_intake.__file__}')

/Users/leonardobertini/Desktop/precog-esgf-intake
Python: /Users/leonardobertini/Desktop/precog-esgf-intake/.venv/bin/python3.12
Environment: /Users/leonardobertini/Desktop/precog-esgf-intake/.venv
Working directory: /Users/leonardobertini/Desktop/precog-esgf-intake

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Imported Packaged Layout for precog-esgf-intake, which points to: /Users/leonardobertini/Desktop/precog-esgf-intake/src/precog_esgf_intake/__init__.py


Ok. Now if you reached this step and saw printed output without any errors, you have configured your Jupyter Notebook
with the right kernel using our Python 3.12 virtual environment.

## Step 4. Create directory to save search results

Now Create a directory named
```test_search```
under your Desktop.


In [3]:
%%bash
mkdir -p ~/Desktop/test_search

PS. This is equivalent to running ```> mkdir ~/Desktop/test_search``` on a ```Terminal prompt```


Run the next cell to display an example TOML layout that can be adapted to your own ESGF search criteria.


### Step 4.1 Edit the TOML search configuration

Before running the ESGF catalogue sweep, edit the user-facing TOML configuration file so that it matches your intended
ESGF search. [An example of a default TOML file can be found here](https://github.com/precog-ocean/precog-esgf-intake/blob/main/scripts/search_criteria.toml).

This makes the workflow more reproducible than changing hard-coded Python dictionaries inside the script.

The TOML file can be used to predefine search facets such as project, activity, experiment, frequency, variables, and grid labels. A typical configuration might include entries such as `project = ['CMIP6']`, `experiment_id = ['piControl', 'historical']`, `frequency = ['mon']`, and `grid_label = ['gn', 'gr']`.

If `variable_id` is intentionally left empty in the TOML file, the catalogue script can still fall back to an interactive prompt so that you can type the variables at run time. This is useful when the rest of the search criteria should remain fixed but the variable list changes between runs.

In [4]:
example_toml = '''
[paths]
download_path = "~/Desktop/test_search"

[esgf]
project = ["CMIP6"]
activity_drs = ["CMIP", "ScenarioMIP"]
experiment_id = ["piControl", "historical"]
frequency = ["mon"]
variable_id = ["expc", "epc100"]
grid_label = ["gn", "gr"]
'''
print(example_toml)



[paths]
download_path = "~/Desktop/test_search"

[esgf]
project = ["CMIP6"]
activity_drs = ["CMIP", "ScenarioMIP"]
experiment_id = ["piControl", "historical"]
frequency = ["mon"]
variable_id = ["expc", "epc100"]
grid_label = ["gn", "gr"]



The template `search_criteria.toml` file that is shipped with ```precog-esfg-intake``` can be modified.
PS. In future implementations, this could be passed as an optional argument when calling
```intake_Catalogue_Search.py``` to accommodate various externally supplied search configurations.
The key point is that the search criteria are now externalised and can be edited before Step 5.


## Step 5. ESGF Catalogue Search

Now run the next cell in the notebook to execute `intake_CatalogueSearch.py` and follow the CLI in-prompt
instructions. This will emulate a terminal window and the command line interface from within your Jupyter Notebook,
so you can interact with it.

<br>

When asked to provide `variable_ids`, insert `['expc', 'epc100']`.


In [5]:
%cd ~/Desktop/precog-esgf-intake
%run src/precog_esgf_intake/scripts/intake_CatalogueSearch.py

/Users/leonardobertini/Desktop/precog-esgf-intake



         5PO8$HHDBWWWWBBHK@UOPp                                                                                         
     !P&KWN0QQQQQQQ000QQQQQQ0NWHUGn                                                                                     
   FAHMQQQ0000000000000RR00000QQQ0D&g                                                                                   
 IbBQQ00000000000000Q0000R00000QXPAUY4gg4SShc  '2SVgggg4SSh%  ;pSdgggggggg>  +mS4ggggggu   7ghggggggSg7   Lghggggggggt  
jAQQR0000000000000Q0000RRR00000HnMQPezjz3QQN/  aQQG77zz3QQW+  jQQEo77777jj,  TQQELLLLLLx  <BQNnLLLnNQQI  %MQDuLj7777j>  
EQ0RRR0RRMNMMRRRRR00Q0Q000000RQ2ZQQ4T#ywEQQj  ,OQQVTJyJ4QQz  +$QQST#CCCCv   \HQD^         FQQj    *0Q4   gQQ1  "Lzzj)   
XWDDDHDBDWWNWBWBBDHKHDDDBHDDDWkJQQV}?1[1I*{-  TQQh}?!7QQW?-  fQQF{*}***}=   5QQj         lRQA`   ,OQR)  I0QZ   )aQQQ>   
fGK@KgEHdOK@K4GK@KkpV@$$Uq8@U$TORD| 1Fmw     |$0D/   |N0@_  )H00dppmmmmF+  xW0RVqhGPGz  -SQ0AmqPPXQ0

Define the download path. Either type path or drag desired download parent folder onto this terminal window: /Users/leonardobertini/Desktop/test_search




======= Printing config. set up for the upcoming ESGF search =======
additional_df_cols: []
break_on_error: true
confirm_download: true
download_db: ~/.config/intake-esgf/download.db
esg_dataroot:
- /p/css03/esgf_publish
- /eagle/projects/ESGF2/esg_dataroot
- /global/cfs/projectdirs/m3522/cmip6/
- /glade/campaign/collections/cmip.mirror
globus_indices:
  ESGF2-US-1.5-Catalog: true
  anl-dev: false
  ornl-dev: false
local_cache:
- !!python/object/apply:pathlib.PosixPath
  - /
  - Users
  - leonardobertini
  - Desktop
  - test_search
logfile: ~/.config/intake-esgf/esgf.log
num_threads: 6
print_log_on_error: false
requests_cache:
  cache_name: intake-esgf/requests-cache.sqlite
  expire_after: 3600
  use_cache_dir: true
slow_download_threshold: 0.5
solr_indices:
  esg-dn1.nsc.liu.se: false
  esgf-data.dkrz.de: false
  esgf-node.ipsl.upmc.fr: false
  esgf-node.llnl.gov: false
  esgf-node.ornl.gov: false
  esgf.ceda.ac.uk: false
  esgf.nci.org.au: false
stac_indices:
  api.stac.ceda.ac.uk:

No variable_id values were provided in search_criteria.toml.
Please enter a list of strings for variable_id. E.g. ['expc'] or multiple ['expc', 'o2', 'thetao', 'so', 'o2sat', 'epc100']:
 ['expc', 'epc100']


User entered variable_ids: ['expc', 'epc100']
======== Running ESGF search with the following resolved criteria ======== 
project: ['CMIP6']
activity_drs: ['CMIP', 'ScenarioMIP']
experiment_id: ['piControl', 'historical']
frequency: ['mon']
variable_id: ['expc', 'epc100']
grid_label: ['gn', 'gr']



Choose ensemble handling mode:
'single' = this will return members across piControl and historical runs (sharing the exact same variant label)
'ensemble' = lowest piControl variant as an anchor + all compatible historical variants
Type [single/ensemble]:  single


... (10 outputs left out)




                                                                                
                                )\l!                                            
                              *{1f[{{                                           
                      sll!*{ls{!Dps{{   sxI?{}                                  
                     *{1E5}{rr**pXfI{{x*{2Et{{                                  
                     *{CMjrs*  {r]Dg{{l*?$h{r}                                  
                     s{{zVX[{*>{*fht{{ *{!2bn{{                                 
                      r{swDo{* {*}cl!   *s[Uq{{                                 
                      {{IC1s}           {{ju{{}                                 
            ?**|//////==\"|;'///////////;\|//'=//////l{x                        
            r**#yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyw?{s                        
            r*}KQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQQ01rl%)\/                    
            r*}$QQQQQQQQQ

  0%|          | 0/518 [00:00<?, ?it/s]

2026-11-25 12:32:42,787 - INFO     - Traversing complete for vars ['expc', 'epc100']

2026-11-25 12:32:42,796 - INFO     - Data sweep complete.
Dataframes should have been saved at /Users/leonardobertini/Desktop/test_search
2026-11-25 12:32:42,797 - INFO     - Now giddy up and run "~/intake_OceanVarsDL.py" to download the data.


                  gCLe[]!]1te7zLnTCyFhHQ                                        
                Bfz?i||||||)\<>i%cr{}[[1eznC2gH.                                
              .6u[i|||||||||||||||||)[e[}rcl{I17TyFQ                            
             QC7s|||||||||||||||||||||<x*][Ix<|)ireCQ                           
            Xnai||v\||||||||||||||||||||||>l!1?x\<tTQ                           
           gL]\||{yv|||||||||||||||||||||||||>s]I}oP                            
          qz?|||<jo)|||||||||||||||||||||||||||\rs#Q            .@Xm5233y54     
         Ej?||||?e}||||||||||||||||||||||||||||||{LV          gynt??!??!1ej#FD  
        Qno|||||a]x||||||||||||||||||||||||||%lea%?A        OFj{)c**{lcccclIo5  
        Q#a?*rx**?i|||||||||||)<)vvi<)))))\%s?/:e1}X      Qp#]\%tjfEXVd4gg4nod  
          O63fJCs?l|||||||||vl!er^,;?1v>x*}r))?x[[{zQ   QFnax\{renj7aj]?![aox[b 
                Wjosv>)||||>elalJ/`,[t>vri<\)%!t?{*owJTuo[Ir}[}*1**z"e{vxxxv=:<3
                 Qqyn7t[!I{I

PS. This is equivalent to running the command `> python3.12 src/precog_esgf_intake/scripts/intake_CatalogueSearch.py` on a `Terminal prompt`.

### User prompts

The catalogue sweep supports two related screening modes:

- **Single-branch mode** keeps one internally consistent pairing of `piControl` and `historical` files for each shortlisted model and grid.
- **Ensemble-branch mode** keeps one validated `piControl` anchor branch and then retains all compatible `historical`
 ensemble members that share the same model and grid and pass the variable availability and continuity checks.

Both modes first test whether all requested variables are available for `piControl` and `historical`, and whether
they coexist on at least one common grid label such as `gn` or `gr`.

Type `single` to enter the single-branch mode.

Then in the next prompt type `yes` to perform a union search and insert `['expc', 'epc100']`.

### Search Logs

You'll see the search log printed to screen, which is also saved for future reference under your
`test_search` directory.

For instance, under 'single-branch' mode the log shows that complete PI and
Historical runs for both  `epc100` and `expc` where found for the following CMIP6 models:
```
- GFDL-ESM4
- GISS-E2-1-G
- IPSL-CM6A-LR
- MPI-ESM-1-2-HAM
- MPI-ESM1-2-HR
- MPI-ESM1-2-LR
- UKESM1-0-LL
```

If you inspect the printouts (or the log file produced), you will see that it shows human-readable grid-level
diagnostics in table-like formats. For example, a model may have `expc` available on `gn` for `piControl` but fail the corresponding `historical` test on that same grid, in which case that grid is rejected from downstream screening. For example, model `GISS-E2-1-G-CC` has `expc` available in a native grid (i.e., `gn`) but is lacking `expc` outputs for the Historical run in the native grid.

```
No complete set of variables for model GISS-E2-1-G-CC for variable ['expc', 'epc100'] in either grid. Test returned:
INFO - ####################################################################################################
INFO - grid_label       var_test   variable_ids  has_all_variables        run          model
INFO -         gn  [True, False] [expc, epc100]              False  piControl GISS-E2-1-G-CC
INFO -         gr [False, False] [expc, epc100]              False  piControl GISS-E2-1-G-CC
INFO -         gn  [False, True] [expc, epc100]              False historical GISS-E2-1-G-CC
INFO -         gr [False, False] [expc, epc100]              False historical GISS-E2-1-G-CC
INFO - ####################################################################################################
```

### Files produced

Now look at the `path` you indicated (i.e., `test_search/`) and inspect the files created. You should have the
following:

- `ESGF_search_<datetime_stamp>.xlsx` ==> a dataframe with the raw search results returned from ESGF nodes.
- `ESGF_search_<datetime_stamp>_<varstamp>.log` ==> a text log containing the results of the ESGF sweep together with variable-completeness, grid-consistency, and continuity checks.
- `DF_Downloadable_<varstamp>.xlsx` ==> a filtered dataframe of downloadable rows that passed the screening tests.
- `DF_Downloadable_Ensemble_<varstamp>.xlsx`, if you had run the search under `ensemble` mode.

### Step 5.1 - Manipulating the search results dataframes

The dataframe `DF_Downloadable_expc_epc100.xlsx` contains the rows that survived the catalogue screening.

You can add custom filtering before passing `DF_Downloadable_<varstamp>.xlsx` to the downloader program
`intake_OceanVarsDL.py`, which we will be using in **Step 6**.

For instance, you could create a second dataframe containing only `UKESM1-0-LL` entries by running the following subprogram:

In [8]:
import pandas as pd
from pathlib import Path

def filter_model(path, sheet_name, model):
    # Read the sheet into a DataFrame
    df = pd.read_excel(path, sheet_name=0)  # sheet_name=0 = first sheet
    fname = path.name.split('.')[0] + '_' + model + '.xlsx'
    # Keep only the model you want
    df_model = df[df['source_id'] == model]
    # Export filtered file
    df_model.to_excel(os.path.join(path.parent, fname), sheet_name=sheet_name, index=False)
    print(f"File {path} has been trimmed.")

if __name__ == "__main__":
    # Path to your file
    excel_path = input("Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results:")
    excel_path = Path(excel_path.strip(" "))  # strip needed as dragging onto terminal adds a trailing 'space'
    filter_model(excel_path, sheet_name='Sheet1', model='UKESM1-0-LL')


Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results: /Users/leonardobertini/Desktop/test_search/DF_Downloadable_expc_epc100.xlsx


File /Users/leonardobertini/Desktop/test_search/DF_Downloadable_expc_epc100.xlsx has been trimmed.


For the sake of illustration, you can use the same logic from the subprogram above so that it will delete all rows
but the top 5 on this `UKESM1-0-LL` dataframe. Otherwise, it may prompt you to accept a large download to your disk once you run the program that will fetch data in **Step 6**.

You can trim the dataframe further by running this subprogram below:


In [9]:
import pandas as pd
from pathlib import Path

def keep_top_five_rows(path: Path, sheet_name):
    # Read the sheet into a DataFrame
    df = pd.read_excel(path, sheet_name=0)  # sheet_name=0 = first sheet
    # Keep only the first 5 rows
    df_top5 = df.head(5)
    # Overwrite the original file (no index column)
    df_top5.to_excel(path, sheet_name=sheet_name, index=False)
    print(f"File {path} has been trimmed.")

if __name__ == "__main__":
    # Path to your file
    excel_path = input("Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results:")
    excel_path = Path(excel_path.strip(" "))  # strip needed as dragging onto terminal adds a trailing 'space'
    keep_top_five_rows(excel_path, sheet_name='Sheet1')


Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results: /Users/leonardobertini/Desktop/test_search/DF_Downloadable_expc_epc100_UKESM1-0-LL.xlsx


File /Users/leonardobertini/Desktop/test_search/DF_Downloadable_expc_epc100_UKESM1-0-LL.xlsx has been trimmed.


PS. Alternatively, you can do the same screening by opening the ```DF_Downloadable_<varstamp>.xls``` dataframe on any
spreadsheet processors (like Excel for instance), and then save a refined version of this Dataframe and then pass it
onto the downloader script in **Step 6**.

## Step 6. Fetch the data

The next step is to run the downloader script, which will download the filtered search results from the `ESGF_search_<varstamp>.xlsx` dataframe.

You can indicate where you'd like files to be downloaded to or keep `~/Desktop/search_results` created in **Step 4**
as your default.

Downloads will trigger in parallel, and files will be organised under a directory tree that has a directory named `CMIP6` at the top.

Run the following cell:


In [ ]:
%run src/precog_esgf_intake/scripts/intake_OceanVarsDL.py





         5PO8$HHDBWWWWBBHK@UOPp                                                                                         
     !P&KWN0QQQQQQQ000QQQQQQ0NWHUGn                                                                                     
   FAHMQQQ0000000000000RR00000QQQ0D&g                                                                                   
 IbBQQ00000000000000Q0000R00000QXPAUY4gg4SShc  '2SVgggg4SSh%  ;pSdgggggggg>  +mS4ggggggu   7ghggggggSg7   Lghggggggggt  
jAQQR0000000000000Q0000RRR00000HnMQPezjz3QQN/  aQQG77zz3QQW+  jQQEo77777jj,  TQQELLLLLLx  <BQNnLLLnNQQI  %MQDuLj7777j>  
EQ0RRR0RRMNMMRRRRR00Q0Q000000RQ2ZQQ4T#ywEQQj  ,OQQVTJyJ4QQz  +$QQST#CCCCv   \HQD^         FQQj    *0Q4   gQQ1  "Lzzj)   
XWDDDHDBDWWNWBWBBDHKHDDDBHDDDWkJQQV}?1[1I*{-  TQQh}?!7QQW?-  fQQF{*}***}=   5QQj         lRQA`   ,OQR)  I0QZ   )aQQQ>   
fGK@KgEHdOK@K4GK@KkpV@$$Uq8@U$TORD| 1Fmw     |$0D/   |N0@_  )H00dppmmmmF+  xW0RVqhGPGz  -SQ0AmqPPXQ0F  _E00kmqGX400w    
IpgVqd44dd44ddd4d6qd44hmhd44g

Define the download path. Either type path or drag desired download folder onto this terminal window: /Users/leonardobertini/Desktop/test_search
Now either drag onto terminal or type path to Dataframe with the Filtered ESGF search results desired: /Users/leonardobertini/Desktop/test_search/DF_Downloadable_expc_epc100_UKESM1-0-LL.xlsx


2026-11-25 12:37:56,934 - INFO     - Checking if Dataframe is readable
2026-11-25 12:37:56,936 - INFO     - True
2026-11-25 12:37:56,936 - INFO     - Importing Dataframe DF_Downloadable_expc_epc100_UKESM1-0-LL.xlsx
2026-11-25 12:37:56,952 - INFO     - There are 5 files totalling 37.68 Gb for variable(s) ['expc']


Do you want to continue to downloads? (yes/no):  yes


2026-11-25 12:38:10,596 - INFO     - User said yes
2026-11-25 12:38:10,597 - INFO     - Continuing...

2026-11-25 12:38:12,170 - INFO     - Starting download for file: expc_Omon_UKESM1-0-LL_piControl_r1i1p1f2_gn_196001-199912.nc
2026-11-25 12:38:12,179 - INFO     - Starting download for file: expc_Omon_UKESM1-0-LL_piControl_r1i1p1f2_gn_200001-204912.nc


  0%|          | 0.00/5.84G [00:00<?, ?B/s]

  0%|          | 0.00/7.31G [00:00<?, ?B/s]

PS. This is equivalent to running the command `> python3.12 src/precog_esgf_intake/intake_OcanVarsDL.py` on a `Terminal prompt`.

When the program finishes running, a folder `CMIP6` should have been created within your `download_path` with the data organised per model.


## Step 7. Fetch Grid cell measures (`areacello` and `volcello`)

Run the next script to fetch corresponding grid cell measures `areacello` and `volcello` for the downloaded ESM outputs.

The program will import automatically the variables from *all* the available dataframes `DF_Downloadable_<varstamp>
.xlsx` residing in the parent directory that you indicate. If you'd like a specific dataframe to spawn the cell measure
search, then provide the full path to that specific `DF_Downloadable_<varstamp>.xlsx` dataframe.

Then the program will create a new dataframe `DF_Downloadable_<cellmeasure_stamp>.xlsx` on the chosen
`download_path`, with the cell measure search results.

Then you'll be prompted to indicate the path to this newly created dataframe `DF_Downloadable_<cellmeasure_stamp>.xlsx`, and the cell measure downloads will trigger in parallel.

Files will be organised under the same directory tree, with a directory `CMIP6` at the top.

Run the following:

In [ ]:
%run src/precog_esgf_intake/scripts/intake_CellMeasuresDL.py


PS. This is equivalent to running the command `> python src/precog_esgf_intake/scripts/intake_CellMeasuresDL.py` on a `Terminal prompt`.


## Step 8. Check files

That's it. Now inspect `download_path` to check the directory tree and if the files were downloaded.
